<a href="https://colab.research.google.com/github/vyakhyaagoyal/aiml_colab_sem-4/blob/main/gradient_descent_path.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Checkbox
from mpl_toolkits.mplot3d import Axes3D

# -------------------------------------------------
# Example dataset
# -------------------------------------------------
data = pd.DataFrame({
    "x": [1, 2, 3, 5],
    "y": [4, 6, 7, 15]
})

x_data = data["x"].values
y_data = data["y"].values
m = len(x_data)

# -------------------------------------------------
# Hypothesis function
# -------------------------------------------------
def h(theta0, theta1, x):
    return theta0 + theta1 * x

# -------------------------------------------------
# Cost function
# -------------------------------------------------
def compute_cost(theta0, theta1, x, y):
    m = len(x)
    y_pred = theta0 + theta1 * x
    return (1 / (2 * m)) * np.sum((y_pred - y) ** 2)

# -------------------------------------------------
# Gradients
# -------------------------------------------------
def compute_gradients(theta0, theta1, x, y):
    m = len(x)
    y_pred = theta0 + theta1 * x
    error = y_pred - y

    dtheta0 = (1 / m) * np.sum(error)
    dtheta1 = (1 / m) * np.sum(error * x)

    return dtheta0, dtheta1

# -------------------------------------------------
# Gradient Descent path
# -------------------------------------------------
def gradient_descent_path(x, y, theta0_init, theta1_init, alpha, num_iters):
    theta0 = theta0_init
    theta1 = theta1_init

    theta0_history = [theta0]
    theta1_history = [theta1]
    J_history = [compute_cost(theta0, theta1, x, y)]

    for _ in range(num_iters):
        dtheta0, dtheta1 = compute_gradients(theta0, theta1, x, y)

        theta0 = theta0 - alpha * dtheta0
        theta1 = theta1 - alpha * dtheta1

        theta0_history.append(theta0)
        theta1_history.append(theta1)
        J_history.append(compute_cost(theta0, theta1, x, y))

    return theta0_history, theta1_history, J_history

# -------------------------------------------------
# Precompute contour/surface grid
# -------------------------------------------------
theta0_vals = np.linspace(-5, 10, 120)
theta1_vals = np.linspace(-1, 5, 120)

T0, T1 = np.meshgrid(theta0_vals, theta1_vals)
J_vals = np.zeros(T0.shape)

for i in range(T0.shape[0]):
    for j in range(T0.shape[1]):
        J_vals[i, j] = compute_cost(T0[i, j], T1[i, j], x_data, y_data)

# -------------------------------------------------
# Interactive visualization
# -------------------------------------------------
def visualize(theta0=0.0, theta1=1.0, show_gd_path=True, alpha=0.08, num_iters=20):
    J_current = compute_cost(theta0, theta1, x_data, y_data)

    # Gradient descent path starting from current slider values
    theta0_hist, theta1_hist, J_hist = gradient_descent_path(
        x_data, y_data, theta0, theta1, alpha, num_iters
    )

    fig = plt.figure(figsize=(18, 10))

    # ---------------------------------------------
    # 1. Regression line
    # ---------------------------------------------
    ax1 = fig.add_subplot(2, 2, 1)
    ax1.scatter(x_data, y_data, label="Actual Data")
    x_line = np.linspace(min(x_data) - 0.5, max(x_data) + 0.5, 100)
    ax1.plot(x_line, h(theta0, theta1, x_line), label="Tuned Line")
    ax1.set_xlabel("x")
    ax1.set_ylabel("y")
    ax1.set_title(f"Regression Line\nCurrent Cost J = {J_current:.4f}")
    ax1.legend()
    ax1.grid(True)

    # ---------------------------------------------
    # 2. Contour plot
    # ---------------------------------------------
    ax2 = fig.add_subplot(2, 2, 2)
    contour = ax2.contour(T0, T1, J_vals, levels=30)
    ax2.clabel(contour, inline=True, fontsize=8)

    # Current theta point
    ax2.plot(theta0, theta1, 'ro', markersize=8, label="Current theta")

    # Optional GD path
    if show_gd_path:
        ax2.plot(theta0_hist, theta1_hist, 'bo-', linewidth=2, markersize=4, label="GD Path")
        ax2.plot(theta0_hist[0], theta1_hist[0], 'ks', markersize=8, label="Start")
        ax2.plot(theta0_hist[-1], theta1_hist[-1], 'g*', markersize=12, label="GD End")

    ax2.set_xlabel(r'$\theta_0$')
    ax2.set_ylabel(r'$\theta_1$')
    ax2.set_title(r'2D Contour Plot of $J(\theta_0,\theta_1)$')
    ax2.legend()
    ax2.grid(True)

    # ---------------------------------------------
    # 3. 3D Surface plot
    # ---------------------------------------------
    ax3 = fig.add_subplot(2, 2, 3, projection='3d')
    ax3.plot_surface(T0, T1, J_vals, alpha=0.65)

    # Current theta point
    ax3.scatter(theta0, theta1, J_current, color='red', s=60, label='Current theta')

    # Optional GD path
    if show_gd_path:
        ax3.plot(theta0_hist, theta1_hist, J_hist, 'bo-', linewidth=2, markersize=4)
        ax3.scatter(theta0_hist[0], theta1_hist[0], J_hist[0], color='black', s=50)
        ax3.scatter(theta0_hist[-1], theta1_hist[-1], J_hist[-1], color='green', s=80)

    ax3.set_xlabel(r'$\theta_0$')
    ax3.set_ylabel(r'$\theta_1$')
    ax3.set_zlabel(r'$J(\theta)$')
    ax3.set_title(r'3D Surface Plot of $J(\theta_0,\theta_1)$')

    # ---------------------------------------------
    # 4. Cost vs iteration
    # ---------------------------------------------
    ax4 = fig.add_subplot(2, 2, 4)
    if show_gd_path:
        ax4.plot(range(len(J_hist)), J_hist, 'mo-', linewidth=2, markersize=4)
        ax4.set_title("Cost vs Iteration (from current slider theta)")
    else:
        ax4.bar(["Current Cost"], [J_current])
        ax4.set_title("Current Cost Only")

    ax4.set_xlabel("Iteration")
    ax4.set_ylabel(r'$J(\theta)$')
    ax4.grid(True)

    plt.tight_layout()
    plt.show()

    print(f"Current theta0 = {theta0:.3f}")
    print(f"Current theta1 = {theta1:.3f}")
    print(f"Current cost   = {J_current:.6f}")

    if show_gd_path:
        print(f"GD final theta0 = {theta0_hist[-1]:.6f}")
        print(f"GD final theta1 = {theta1_hist[-1]:.6f}")
        print(f"GD final cost   = {J_hist[-1]:.6f}")

# -------------------------------------------------
# Sliders
# -------------------------------------------------
interact(
    visualize,
    theta0=FloatSlider(min=-5, max=10, step=0.1, value=0.0, description='theta0'),
    theta1=FloatSlider(min=-1, max=5, step=0.1, value=1.0, description='theta1'),
    show_gd_path=Checkbox(value=True, description='Show GD Path'),
    alpha=FloatSlider(min=0.01, max=0.5, step=0.01, value=0.08, description='alpha'),
    num_iters=IntSlider(min=1, max=50, step=1, value=20, description='iters')
);

interactive(children=(FloatSlider(value=0.0, description='theta0', max=10.0, min=-5.0), FloatSlider(value=1.0,…